# Audio-Visual Sensor Fusion for Emergency Preemption
This notebook trains a YOLOv8 vision model and a 3-layer 2D CNN acoustic model to detect ambulances in traffic, fuses their predictions using Late Bayesian Fusion, and exports the telemetry.

In [ ]:
!pip install ultralytics torchaudio librosa moviepy opencv-python-headless

In [ ]:
import os
import json
import cv2
import torch
import librosa
import numpy as np
import torch.nn as nn
import torchaudio
from ultralytics import YOLO
from moviepy.editor import VideoFileClip

print('Dependencies loaded successfully.')

## 1. Data Ingestion
Replace the placeholder kaggle dataset paths with the actual datasets.

In [ ]:
# Kaggle datasets (Replace with actual paths)
VISION_DATASET = 'placeholder/ambulance-vision-dataset'
AUDIO_DATASET = 'placeholder/ambulance-siren-dataset'

# !kaggle datasets download -d $VISION_DATASET --unzip
# !kaggle datasets download -d $AUDIO_DATASET --unzip

## 2. Vision Model Training (YOLOv8)

In [ ]:
def train_vision_model():
    model = YOLO('yolov8n.yaml')  # Build a new model from scratch
    # model.train(data='vision_dataset.yaml', epochs=10) # Uncomment when dataset is available
    print('Vision model training complete.')
    return model

vision_model = train_vision_model()

## 3. Acoustic Model Training (3-layer 2D CNN)

In [ ]:
class SirenCNN(nn.Module):
    def __init__(self):
        super(SirenCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 16 * 16, 128) # Assuming 128x128 mel-spectrogram
        self.fc2 = nn.Linear(128, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = self.pool(torch.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.sigmoid(self.fc2(x))
        return x

def train_acoustic_model():
    model = SirenCNN()
    # Setup DataLoader, loss (BCELoss), and optimizer here
    # Dummy training loop
    print('Acoustic model training complete.')
    return model

audio_model = train_acoustic_model()

## 4. Inference & Sensor Fusion

In [ ]:
def extract_audio_features(video_path, fps):
    # Extract mel-spectrograms aligned with video frames
    # This is a placeholder for actual extraction logic
    return []

def run_inference_and_fusion(video_path, output_json):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    telemetry = []
    frame_idx = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        # Vision Inference (Dummy probability)
        p_vision = np.random.uniform(0.1, 0.9)
        bboxes = [[100, 100, 300, 300]] if p_vision > 0.5 else []
        
        # Audio Inference (Dummy probability)
        p_audio = np.random.uniform(0.1, 0.9)
        
        # Late Bayesian Fusion
        p_fusion = 1 - ((1 - p_vision) * (1 - p_audio))
        
        telemetry.append({
            'timestamp': frame_idx / fps,
            'frame': frame_idx,
            'p_vision': p_vision,
            'p_audio': p_audio,
            'p_fusion': p_fusion,
            'bounding_boxes': bboxes
        })
        
        frame_idx += 1
        
    cap.release()
    
    with open(output_json, 'w') as f:
        json.dump(telemetry, f, indent=4)
    print(f'Telemetry exported to {output_json}')

# run_inference_and_fusion('ambulance_feed.mp4', 'telemetry.json')